# expert-priors quickstart

From a raw table to LLM-elicited priors to prior-informed models, benchmarked **in-domain and out-of-domain** — in a few lines.

**No API key needed to run this notebook:** the priors were elicited once (with `claude-opus-4-8`) and cached in `examples/priors/housing_prior.json`. To regenerate them yourself, run `python examples/build_demo_priors.py` with an API key.

### The setup
A small synthetic housing dataset with a deliberate **domain shift**. In-domain (ID) homes are nearly new (age 0–4 yrs), so the depreciation effect is barely visible in the training data. Out-of-domain (OOD) homes are old (age 6–40 yrs), where age matters a lot. The training data says little about the age slope, so how a model extrapolates to old homes depends on what it shrinks toward: zero for a data-only baseline, or the LLM's belief that older houses are worth less.

In [1]:
import expert_priors as ep
from expert_priors.bundle import PriorBundle
from demo_data import make_housing  # local helper in examples/

X, y, ood_mask = make_housing()
print(f"{len(X)} homes  |  {int(ood_mask.sum())} out-of-domain (old)")
X.head()

210 homes  |  60 out-of-domain (old)


,sqft,bedrooms,age_years,garage
0,2116.0,2.0,3.7,1.0
1,1546.0,5.0,0.1,0.0
2,2260.0,1.0,2.2,1.0
3,1986.0,5.0,2.5,1.0
4,960.0,3.0,0.4,0.0


## 1. Load the elicited priors
Each row is one coefficient draw (in raw USD-per-unit) from the LLM's belief distribution. The mean already looks sensible: positive for size/bedrooms/garage, **negative for age** (depreciation).

In [2]:
priors = PriorBundle.load("priors/housing_prior.json")
print(priors)
dict(zip(priors.feature_names, priors.mean().round(0).tolist()))

PriorBundle(task='regression', model='claude-opus-4-8', n_samples=20, n_features=4)


{'sqft': 150.0, 'bedrooms': 9350.0, 'age_years': -3350.0, 'garage': 13800.0}

> To elicit priors for **your own** data instead, it's one call — and the LLM can even write the feature descriptions for you:
>
> ```python
> priors = ep.elicit_priors(X, y, task="regression",
>                           model="claude-opus-4-8", auto_context=True,
>                           cache="priors/mydata.json")
> ```

## 2. Fit a prior-informed model
`PriorMeanRidge` is a drop-in scikit-learn estimator: it fits on raw `X` and exposes raw-space `coef_`. `alpha="cv"` tunes the prior strength by cross-validation. We fit on the in-domain homes only.

In [3]:
X_id, y_id = X[~ood_mask], y[~ood_mask]
model = ep.PriorMeanRidge(priors, alpha="cv").fit(X_id, y_id)
dict(zip(priors.feature_names, model.coef_.round(0).tolist()))

{'sqft': 152.0, 'bedrooms': 14147.0, 'age_years': -1028.0, 'garage': 18933.0}

## 3. Benchmark baseline vs prior families (in-domain & out-of-domain)
`compare()` fits the L2 baseline (Ridge) plus every prior family, trains on in-domain homes, and evaluates on held-out in-domain **and** out-of-domain homes.

In [4]:
report = ep.compare(X, y, priors, ood_mask=ood_mask, alpha="cv")
report.summary().round(1)[["model", "split", "n", "rmse", "mse", "r2"]]

,model,split,n,rmse,mse,r2
0,Ridge,ID,38,8111.3,65793359.6,1.0
1,Ridge,OOD,60,20115.7,404640211.8,0.9
2,Prior-Mean,ID,38,8112.6,65814844.9,1.0
3,Prior-Mean,OOD,60,19579.7,383365704.2,0.9
4,Correlated,ID,38,8163.9,66649166.3,1.0
5,Correlated,OOD,60,21106.4,445479377.4,0.9
6,Independent,ID,38,8156.9,66534718.9,1.0
7,Independent,OOD,60,21130.7,446507283.5,0.9


In [5]:
# Signed lift on the primary metric (MSE); positive = better than the Ridge baseline
report.lift().round(1)[["split", "model", "lift_pct"]]

,split,model,lift_pct
0,ID,Prior-Mean,-0.0
1,ID,Correlated,-1.3
2,ID,Independent,-1.1
3,OOD,Prior-Mean,5.3
4,OOD,Correlated,-10.1
5,OOD,Independent,-10.3


## Takeaway
In-domain, every model performs about the same: the training data already pins the coefficients, and cross-validation keeps the prior from overriding it. Out-of-domain, the models differ in how they extrapolate the coefficients the data could not pin down, and that is where the prior family matters.

`compare()` gives you this ID/OOD view on your own data in one call, so you can see which prior family suits your problem.